# M3 Notebook 05 — Regularization and Generalization

**Status:** Runnable first edition

## Learning objectives

- Apply L2 regularization, dropout, and early stopping.
- Diagnose overfitting with learning curves.
- Understand capacity and generalization.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_dl import (
    Activation,Adam,Dense,Dropout,Sequential,early_stopping,l2_penalty,
    train_regression,
)


In [ ]:
rng=np.random.default_rng(5)
X=np.linspace(-3,3,120)[:,None]
y=(np.sin(X[:,0])+rng.normal(scale=.25,size=120))[:,None]
train_idx=np.arange(0,80)
val_idx=np.arange(80,120)

def make_net(dropout=0.0,seed=5):
    layers=[Dense(1,32,seed=seed),Activation("tanh")]
    if dropout>0:
        layers.append(Dropout(dropout,seed=seed+10))
    layers += [Dense(32,32,seed=seed+1),Activation("tanh"),
               Dense(32,1,seed=seed+2,scale=.1),Activation("linear")]
    return Sequential(layers)

configs={
    "No regularization":(make_net(0,5),0.0),
    "L2":(make_net(0,15),1e-3),
    "Dropout":(make_net(.2,25),0.0),
}
results={}
for name,(net,l2) in configs.items():
    train_hist=train_regression(net,Adam(.01),X[train_idx],y[train_idx],epochs=1500,l2=l2)
    val_pred=net.predict(X[val_idx])
    val_mse=float(np.mean((val_pred-y[val_idx])**2))
    results[name]=(train_hist,val_mse,net)

pd.DataFrame([
    {"configuration":name,"train_MSE":hist[-1],"validation_MSE":val}
    for name,(hist,val,net) in results.items()
])


In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
for name,(hist,val,net) in results.items():
    ax.semilogy(hist,label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Training MSE")
ax.set_title("Regularization Comparison"); ax.legend()
plt.show()


## Early stopping

In [ ]:
synthetic_validation=list(np.linspace(1,.2,80))+list(np.linspace(.21,.5,40))
stop_epoch=early_stopping(synthetic_validation,patience=8,min_delta=.001)
stop_epoch


## Generalization controls

Regularization reduces effective capacity, but the best method depends on data size, noise, architecture, and validation design.

## Key insight

Generalization is controlled by the interaction of data, model capacity, optimization, regularization, and evaluation.